In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder



In [ ]:
data = pd.read_csv("/content/drive/MyDrive/DataSets/archive (4)/Students Performance .csv")
data.head()

,Student_ID,Student_Age,Sex,High_School_Type,Scholarship,Additional_Work,Sports_activity,Transportation,Weekly_Study_Hours,Attendance,Reading,Notes,Listening_in_Class,Project_work,Grade
0,STUDENT1,19-22,Male,Other,50%,Yes,No,Private,0,Always,Yes,Yes,No,No,AA
1,STUDENT2,19-22,Male,Other,50%,Yes,No,Private,0,Always,Yes,No,Yes,Yes,AA
2,STUDENT3,19-22,Male,State,50%,No,No,Private,2,Never,No,No,No,Yes,AA
3,STUDENT4,18,Female,Private,50%,Yes,No,Bus,2,Always,No,Yes,No,No,AA
4,STUDENT5,19-22,Male,Private,50%,No,No,Bus,12,Always,Yes,No,Yes,Yes,AA


In [ ]:
X = data.drop(['Student_ID', 'Grade'], axis=1)
y = data['Grade']


In [ ]:
encoder = LabelEncoder()

for column in X.columns:
    X[column] = encoder.fit_transform(X[column])

y = encoder.fit_transform(y)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)


DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.10344827586206896


In [ ]:
new_student = pd.DataFrame([{
    'Student_Age': '19-22',
    'Sex': 'Male',
    'High_School_Type': 'Private',
    'Scholarship': '50%',
    'Additional_Work': 'No',
    'Sports_activity': 'No',
    'Transportation': 'Bus',
    'Weekly_Study_Hours': 5,
    'Attendance': 'Always',
    'Reading': 'Yes',
    'Notes': 'Yes',
    'Listening_in_Class': 'Yes',
    'Project_work': 'Yes'
}])

for column in new_student.columns:
    new_student[column] = encoder.fit_transform(new_student[column])

prediction = model.predict(new_student)
print("Predicted Grade:", prediction[0])


Predicted Grade: 0


In [ ]:
!pip install gradio


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
import gradio as gr


In [ ]:
data = {
    'Student_ID': ['STUDENT1','STUDENT2','STUDENT3','STUDENT4','STUDENT5'],
    'Student_Age': ['19-22','19-22','19-22','18','19-22'],
    'Sex': ['Male','Male','Male','Female','Male'],
    'High_School_Type': ['Other','Other','State','Private','Private'],
    'Scholarship': ['50%','50%','50%','50%','50%'],
    'Additional_Work': ['Yes','Yes','No','Yes','No'],
    'Sports_activity': ['No','No','No','No','No'],
    'Transportation': ['Private','Private','Private','Bus','Bus'],
    'Weekly_Study_Hours': [0,0,2,2,12],
    'Attendance': ['Always','Always','Never','Always','Always'],
    'Reading': ['Yes','Yes','No','No','Yes'],
    'Notes': ['Yes','No','No','Yes','No'],
    'Listening_in_Class': ['No','Yes','No','No','Yes'],
    'Project_work': ['No','Yes','Yes','No','Yes'],
    'Grade': ['AA','AA','AA','AA','AA']
}

df = pd.DataFrame(data)


In [ ]:
X = df.drop(['Student_ID', 'Grade'], axis=1)
y = df['Grade']

# Encode categorical data
encoder = LabelEncoder()
for column in X.columns:
    X[column] = encoder.fit_transform(X[column])

y = encoder.fit_transform(y)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, random_state=42)
model.fit(X_train, y_train)


DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, random_state=42)

In [ ]:
def predict_grade(Student_Age, Sex, High_School_Type, Scholarship, Additional_Work,
                  Sports_activity, Transportation, Weekly_Study_Hours, Attendance,
                  Reading, Notes, Listening_in_Class, Project_work):

    # Create a dataframe from input
    new_student = pd.DataFrame([{
        'Student_Age': Student_Age,
        'Sex': Sex,
        'High_School_Type': High_School_Type,
        'Scholarship': Scholarship,
        'Additional_Work': Additional_Work,
        'Sports_activity': Sports_activity,
        'Transportation': Transportation,
        'Weekly_Study_Hours': Weekly_Study_Hours,
        'Attendance': Attendance,
        'Reading': Reading,
        'Notes': Notes,
        'Listening_in_Class': Listening_in_Class,
        'Project_work': Project_work
    }])

    # Encode input using same encoder
    for column in new_student.columns:
        new_student[column] = encoder.fit_transform(new_student[column])

    # Predict
    pred = model.predict(new_student)[0]

    # Convert number back to original grade (AA etc.)
    grade_label = df['Grade'].unique()[pred]

    return grade_label


In [ ]:
iface = gr.Interface(
    fn=predict_grade,
    inputs=[
        gr.Textbox(label="Student Age"),
        gr.Dropdown(choices=["Male","Female"], label="Sex"),
        gr.Dropdown(choices=["Other","State","Private"], label="High School Type"),
        gr.Textbox(label="Scholarship"),
        gr.Dropdown(choices=["Yes","No"], label="Additional Work"),
        gr.Dropdown(choices=["Yes","No"], label="Sports Activity"),
        gr.Dropdown(choices=["Private","Bus"], label="Transportation"),
        gr.Number(label="Weekly Study Hours"),
        gr.Dropdown(choices=["Always","Never"], label="Attendance"),
        gr.Dropdown(choices=["Yes","No"], label="Reading"),
        gr.Dropdown(choices=["Yes","No"], label="Notes"),
        gr.Dropdown(choices=["Yes","No"], label="Listening in Class"),
        gr.Dropdown(choices=["Yes","No"], label="Project Work"),
    ],
    outputs=gr.Textbox(label="Predicted Grade"),
    title="Student Performance Prediction",
    description="Enter student details and get predicted Grade"
)

iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8ded507138c57bf030.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
